# Visualize Subtype Trajectories on Brain

This notebook visualizes subtype trajectories on the brain at a single time point.

**To switch subtypes or time points:** Change `SUBTYPE` and `TIME_POINT` in the configuration cell below, then re-run the visualization cell.


In [8]:

import os
import numpy as np
import pandas as pd
from EMDPM.utils import solve_system
from EMDPM.brain_utils import visualize_brain_region_statistics

import numpy as np
import pandas as pd
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.multitest as sm_multi
%matplotlib qt5


In [9]:
SUBTYPE = 0
TIME_POINT = 10

In [10]:
# Paths
result_dir = "/home/dsemchin/Progression_models_simulations/EMDPM/experiments/qsub_jobs/qsub_results/ppmi_gridsearch_refined_fine"
result_file = os.path.join(result_dir, "PPMI_subtyping_grid_103_lambda_f0p300_lambda_cog0p100_lambda_scalar0p500_lambda_jsd0p100_lambda_beta1p000.npz")
visualizations_dir = os.path.join(result_dir, "visualizations")
os.makedirs(visualizations_dir, exist_ok=True)

# Data paths
data_path = "/data01/bgutman/MRI_data/PPMI/data_ppmi_pd.csv"
connectome_path = "/data01/bgutman/LEGACY/Skoltech/datasets/Connectomes/mean_NORM_con_22.csv"

print(f"Loading candidate 103 results...")
print(f"Visualizing Subtype {SUBTYPE} at {TIME_POINT} years")
print(f"Results will be saved to: {visualizations_dir}\n")


Loading candidate 103 results...
Visualizing Subtype 0 at 10 years
Results will be saved to: /home/dsemchin/Progression_models_simulations/EMDPM/experiments/qsub_jobs/qsub_results/ppmi_gridsearch_refined_fine/visualizations



In [ ]:
# Load results
best_data = np.load(result_file, allow_pickle=True)

# Extract model parameters
cluster_f = best_data["cluster_f"]  # (n_subtypes, n_biomarkers)
final_scalar_K = best_data["final_scalar_K"]
final_s = best_data["final_s"]
n_biomarkers = cluster_f.shape[1]

# Load connectivity matrix
df_K = pd.read_csv(connectome_path)
K = df_K.drop(df_K.columns[0], axis=1).to_numpy()
np.fill_diagonal(K, 0)

# Normalize connectivity matrix
row_sums = K.sum(axis=1)
median_row_sum = np.median(row_sums)
K = K / median_row_sum

# Load biomarker names
df = pd.read_csv(data_path)
biomarker_names = [col for col in df.columns 
                   if col.startswith(('L_', 'R_')) and 
                   col.endswith('_thickavg') and 
                   not col.endswith('_thickavg_resid')]

# Time configuration
t_max = 40
t_span = np.linspace(0, t_max, int(t_max/0.01))

# Get forcing term for this subtype
f_subtype = cluster_f[SUBTYPE]

# Compute full trajectory
x0 = np.zeros(n_biomarkers)
Xtraj_full = solve_system(x0, f_subtype, K, t_span, final_scalar_K)
Xtraj_scaled = Xtraj_full * final_s[:, None]  # (n_biomarkers, n_timepoints)

# Find closest time index and extract trajectory values
t_idx = np.argmin(np.abs(t_span - TIME_POINT))
actual_t = t_span[t_idx]
# Extract trajectory values at this time point
traj_values = Xtraj_scaled[:, t_idx]  # (n_biomarkers,)


Biomarker names loaded: 68 regions



In [14]:
# Create DataFrame for brain visualization
region_stats = pd.DataFrame({
    'P-value': [0.001] * n_biomarkers,  # Set all to significant so they all show
    'FX_size': traj_values
}, index=biomarker_names)

# Create visualization
visualize_brain_region_statistics(
    region_stats,
    colormap='coolwarm',
    cbar_string=f'Subtype {SUBTYPE} - Trajectory at {actual_t:.1f} years',
    p_val_threshold=0.05
)

print(f"\nDone! Visualized Subtype {SUBTYPE} at {actual_t:.1f} years.")


/home/dsemchin/Progression_models_simulations/EMDPM/brain_utils.py:41: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  p = float(stats['P-value'])
/home/dsemchin/Progression_models_simulations/EMDPM/brain_utils.py:42: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  d = float(stats['FX_size'])
File already dowloaded (/home/dsemchin/visbrain_data/example_data/lh.aparc.annot).
File already dowloaded (/home/dsemchin/visbrain_data/example_data/rh.aparc.annot).
BrainObj(name='inflated') created


    Annot file loaded (/home/dsemchin/visbrain_data/example_data/lh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, medialorbitofrontal, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, precuneus, rostralanteriorcingulate, rostralmiddlefrontal, superiorfrontal, superiorparietal, superiortemporal, supramarginal, frontalpole, temporalpole, transversetemporal, insula
    Annot file loaded (/home/dsemchin/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inf

TypeError: setMinimum(self, a0: int): argument 1 has unexpected type 'numpy.float64'